<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z326_Backtesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Backtesting — ARIMA vs HAR vs AutoGluon vs Naive

## La idea

No tenemos los valores reales de `202002` para comparar modelos producto a producto. Pero sí tenemos los 36 meses históricos completos. Podemos hacer un **backtesting**:

```
Entrenamos con: 201701 → 201910  (primeros 34 meses)
Predecimos:     201912           (t+2 desde el último mes de entrenamiento)
Comparamos con: valor real de 201912  (que sí conocemos)
```

Esto nos da el **error real por producto para cada modelo** — sin esperar Kaggle.

## ¿Qué analizamos?

1. **Score global** por modelo (RMSE) — ranking interno sin Kaggle
2. **Error por producto** — ¿qué productos son difíciles para todos los modelos?
3. **Test de McNemar** — ¿la diferencia entre modelos es significativa?
4. **Patrones de error** — ¿los modelos se equivocan en los mismos productos o en distintos?

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle pmdarima
!uv pip install autogluon[all]

In [ ]:
import os
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import chi2, runstest_1samp
from sklearn.linear_model import RidgeCV
from pmdarima import auto_arima
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
  'experimento': 'Backtesting-01',
  'semilla_primigenia': 102191,
  # split: entrenamos hasta este periodo, predecimos t+2
  # 201910 → predice 201912 (valor real conocido)
  'periodo_corte': 201910,
  'periodo_target': 201912,
  'alpha_runs': 0.05
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Preparacion de datos

Separamos el set en:
- **train**: todo hasta `periodo_corte` (inclusive) — lo que "sabe" cada modelo
- **real**: el valor de `periodo_target` — nuestro ground truth interno

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

# split
tb_train = tb_ventas.filter(pl.col("periodo") <= PARAM['periodo_corte'])
tb_real  = tb_ventas.filter(pl.col("periodo") == PARAM['periodo_target']).select(["product_id", "tn"]).rename({"tn": "tn_real"})

productos = tb_apredecir["product_id"].to_list()

print(f"Train: hasta {PARAM['periodo_corte']}  |  {tb_train.height} filas")
print(f"Real : {PARAM['periodo_target']}        |  {tb_real.height} productos")

# 3  Modelo 1 — Naive (mediana 6m)

El baseline más simple. Si ningún modelo lo supera claramente, los datos son intrínsecamente difíciles.

In [ ]:
preds_naive = []

for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    w6  = serie[max(0, len(serie)-6):]
    pred = float(np.median(w6))
    preds_naive.append({'product_id': pid, 'pred_naive': max(pred, 0.0)})

tb_naive = pl.DataFrame(preds_naive)
print("Naive listo")

# 4  Modelo 2 — HAR + Test de Racha

In [ ]:
def build_har_features(serie):
    T = len(serie)
    rows_X, rows_y = [], []
    for t in range(12, T):
        rows_X.append([serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()])
        rows_y.append(serie[t])
    return np.array(rows_X), np.array(rows_y)

def har_predict_next(modelo, serie):
    t = len(serie)
    X = np.array([[serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()]])
    return float(modelo.predict(X)[0])

preds_har = []

for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    fallback = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())

    try:
        _, pvalue = runstest_1samp(serie, cutoff='median')
        tiene_estructura = pvalue < PARAM['alpha_runs']
    except Exception:
        tiene_estructura = False

    if not tiene_estructura:
        pred = fallback
    else:
        try:
            from sklearn.linear_model import LinearRegression
            X, y = build_har_features(serie)
            m = LinearRegression().fit(X, y)
            pred_1 = max(har_predict_next(m, serie), 0.0)
            pred   = max(har_predict_next(m, np.append(serie, pred_1)), 0.0)
        except Exception:
            pred = fallback

    preds_har.append({'product_id': pid, 'pred_har': max(pred, 0.0)})

tb_har = pl.DataFrame(preds_har)
print("HAR listo")

# 5  Modelo 3 — AutoARIMA (dos fases)

In [ ]:
preds_arima = []

for pid in productos:
    print(pid, end=' ')
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    fallback = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())

    try:
        m = auto_arima(serie, seasonal=True, m=12, stepwise=True,
                       suppress_warnings=True, error_action='ignore',
                       max_p=3, max_q=3, max_P=2, max_Q=2,
                       random=True, random_state=PARAM['semilla_primigenia'], n_fits=10)
        forecast = m.predict(n_periods=2)
        pred = max(float(forecast[1]), 0.0)
    except Exception:
        try:
            m = auto_arima(serie, seasonal=False, stepwise=True,
                           suppress_warnings=True, error_action='ignore',
                           random=True, random_state=PARAM['semilla_primigenia'], n_fits=10)
            pred = max(float(m.predict(n_periods=2)[1]), 0.0)
        except Exception:
            pred = fallback

    preds_arima.append({'product_id': pid, 'pred_arima': pred})

tb_arima = pl.DataFrame(preds_arima)
print("\nARIMA listo")

# 6  Modelo 4 — AutoGluon

In [ ]:
tb_train_ts = tb_train.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

ts_data = TimeSeriesDataFrame.from_data_frame(
    tb_train_ts.to_pandas(),
    timestamp_column='timestamp',
    id_column='product_id'
)

modelo_ag = TimeSeriesPredictor(
    prediction_length=2,
    target='tn',
    freq='MS',
    eval_metric='RMSE'
)

modelo_ag.fit(ts_data,
    num_val_windows=2,
    time_limit=3600,
    presets='best_quality',
    random_seed=PARAM['semilla_primigenia']
)

In [ ]:
# el target es 201912 = timestamp 2019-12-01
tb_forecast = modelo_ag.predict(ts_data, random_seed=PARAM['semilla_primigenia'])
tb_forecast_pl = pl.from_pandas(tb_forecast.reset_index())

tb_ag = (
    tb_forecast_pl
    .filter(pl.col('timestamp') == datetime(2019, 12, 1))
    .select(['item_id', 'mean'])
    .rename({'item_id': 'product_id', 'mean': 'pred_ag'})
)
tb_ag = tb_ag.with_columns(
    pl.when(pl.col('pred_ag') < 0).then(0.0).otherwise(pl.col('pred_ag')).alias('pred_ag')
)
print("AutoGluon listo")

# 7  Tabla de errores por producto

Juntamos todas las predicciones con el valor real de `201912` y calculamos el error absoluto por producto para cada modelo.

In [ ]:
tb_errores = (
    tb_real
    .join(tb_naive, on='product_id', how='left')
    .join(tb_har,   on='product_id', how='left')
    .join(tb_arima, on='product_id', how='left')
    .join(tb_ag,    on='product_id', how='left')
)

modelos = ['naive', 'har', 'arima', 'ag']

for m in modelos:
    tb_errores = tb_errores.with_columns(
        (pl.col('tn_real') - pl.col(f'pred_{m}')).abs().alias(f'err_{m}')
    )

display(tb_errores.head(10))

## 7.1 Score global — RMSE por modelo

In [ ]:
print("RMSE por modelo (backtesting sobre 201912):")
print()
for m in modelos:
    rmse = float(np.sqrt((tb_errores[f'err_{m}'] ** 2).mean()))
    print(f"  {m:8s}: {rmse:.4f}")

## 7.2 Test de McNemar — ¿quién gana en cada producto?

Para cada par de modelos aplicamos McNemar: ¿la diferencia en quién predice mejor por producto es estadísticamente significativa?

In [ ]:
def mcnemar(err_a, err_b, nombre_a, nombre_b):
    gana_a = (err_a < err_b).sum()
    gana_b = (err_b < err_a).sum()
    n_10, n_01 = gana_a, gana_b
    if n_10 + n_01 == 0:
        print(f"{nombre_a} vs {nombre_b}: sin discrepancias")
        return
    chi2_stat = (abs(n_10 - n_01) - 1)**2 / (n_10 + n_01)
    pvalue = 1 - chi2.cdf(chi2_stat, df=1)
    ganador = nombre_a if n_10 > n_01 else nombre_b
    sig = "** SIGNIFICATIVO **" if pvalue < 0.05 else "no significativo"
    print(f"{nombre_a} vs {nombre_b}:  {nombre_a} gana {n_10} | {nombre_b} gana {n_01}  →  p={pvalue:.4f}  {sig}  →  {ganador}")

print("Test de McNemar (comparaciones pareadas):")
print()
pares = [('naive','har'), ('naive','arima'), ('naive','ag'),
         ('har','arima'), ('har','ag'), ('arima','ag')]

for a, b in pares:
    ea = tb_errores[f'err_{a}'].to_numpy()
    eb = tb_errores[f'err_{b}'].to_numpy()
    mcnemar(ea, eb, a, b)

## 7.3 ¿Qué productos son difíciles para TODOS los modelos?

In [ ]:
# error promedio entre todos los modelos por producto
tb_errores = tb_errores.with_columns(
    ((pl.col('err_naive') + pl.col('err_har') + pl.col('err_arima') + pl.col('err_ag')) / 4).alias('err_promedio')
)

# top 15 productos mas dificiles
top_dificiles = tb_errores.sort('err_promedio', descending=True).head(15)
display(top_dificiles.select(['product_id', 'tn_real', 'pred_naive', 'pred_har', 'pred_arima', 'pred_ag', 'err_promedio']))

## 7.4 Patron de errores — ¿los modelos se equivocan en los mismos productos?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

pares_plot = [('naive','har'), ('naive','arima'), ('naive','ag'),
              ('har','arima'), ('har','ag'), ('arima','ag')]

for i, (a, b) in enumerate(pares_plot):
    ea = np.log1p(tb_errores[f'err_{a}'].to_numpy())
    eb = np.log1p(tb_errores[f'err_{b}'].to_numpy())

    colores = ['tomato' if ea[j] < eb[j] else 'steelblue' for j in range(len(ea))]
    axes[i].scatter(ea, eb, alpha=0.4, s=12, c=colores)
    lim = max(ea.max(), eb.max()) * 1.05
    axes[i].plot([0, lim], [0, lim], 'k--', linewidth=0.8)
    axes[i].set_xlabel(f'log(1+err_{a})', fontsize=8)
    axes[i].set_ylabel(f'log(1+err_{b})', fontsize=8)
    axes[i].set_title(f'{a} vs {b}\nRojo={a} mejor, Azul={b} mejor', fontsize=8)

fig.suptitle('Patron de errores por producto — backtesting 201912', fontsize=11)
plt.tight_layout()
plt.show()

## 7.5 Visualizacion de los productos mas dificiles

In [ ]:
pids_dificiles = top_dificiles.head(6)['product_id'].to_list()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, pid in enumerate(pids_dificiles):
    serie_full = (
        tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
    )
    ts = serie_full['periodo'].to_list()
    tn = serie_full['tn'].to_numpy()

    row = tb_errores.filter(pl.col('product_id') == pid)
    real   = float(row['tn_real'][0])
    p_naive = float(row['pred_naive'][0])
    p_har   = float(row['pred_har'][0])
    p_arima = float(row['pred_arima'][0])
    p_ag    = float(row['pred_ag'][0])

    axes[i].plot(range(len(tn)), tn, 'o-', color='steelblue', markersize=3, linewidth=1.5, label='real')
    axes[i].axhline(real,    color='black',  linestyle='-',  linewidth=2,   label=f'real 201912={real:.1f}')
    axes[i].axhline(p_naive, color='gray',   linestyle='--', linewidth=1.2, label=f'naive={p_naive:.1f}')
    axes[i].axhline(p_har,   color='green',  linestyle='--', linewidth=1.2, label=f'HAR={p_har:.1f}')
    axes[i].axhline(p_arima, color='orange', linestyle='--', linewidth=1.2, label=f'ARIMA={p_arima:.1f}')
    axes[i].axhline(p_ag,    color='red',    linestyle='--', linewidth=1.2, label=f'AG={p_ag:.1f}')
    axes[i].set_title(f'product_id {pid}', fontsize=9)
    axes[i].legend(fontsize=6)

fig.suptitle('Productos más difíciles — predicciones vs real 201912', fontsize=11)
plt.tight_layout()
plt.show()

# 8  Guardar tabla de errores

Para reutilizar en análisis posteriores o para el McNemar cuando lleguen los datos reales de 202002.

In [ ]:
tb_errores.write_csv('backtesting_errores_201912.csv')
print("Guardado: backtesting_errores_201912.csv")
display(tb_errores.describe())